# Ascend C的“hello world”

本节是Ascend C算子开发的基础章节，将会按照如下结构，带你快速了解Ascend C核函数。

本节学习大纲如下：
- 如何编写核函数
- Ascend C异构并行编程模型
- 如何调用核函数
- 实现Ascend C编程HelloWorld

---
## 1. 环境准备

正式开始学习之前，先要对jupyter环境进行初始化。以下代码完成了初始化并将环境中的变量导入jupyter环境，同时完成了代码目录的创建。保证能正常导入代码以及使用bisheng编译器，完成算子的开发及编译。

In [ ]:
import os
!mkdir -p Sources
!bash -l -c 'source ~/.bashrc && env' > /tmp/shell_env.txt  # bash
with open("/tmp/shell_env.txt", "r") as f:
    for line in f.readlines():
        line = line.strip()
        if "=" in line and not line.startswith(("#", " ")):
            key, value = line.split("=", 1)
            os.environ[key] = value

## 2. 如何编写核函数

核函数（Kernel Function）是Ascend C算子设备侧的入口。Ascend C允许用户使用核函数这种C/C++函数的语法扩展来管理设备侧的运行代码，用户在核函数中进行算子类对象的创建和其成员函数的调用，由此实现该算子的所有功能。核函数是主机端和设备端连接的桥梁。

下图为Ascend C核函数（Kernel）的函数声明：

<img src="./images/核函数.png" alt="核函数"  width="700px" >

### 2.1 修饰符说明

- \_\_global__：表明该函数是可以被主机端（Host）调用的核函数。

- \_\_aicore__：指定该核函数在昇腾NPU处理的AI Core单元上执行。

### 2.2 返回值与参数规则

- 返回值：核函数必须具有 void 返回类型，不能返回任何值。
- 入参类型：仅支持**指针类型**或 C/C++ 内置基础数据类型（如 float*、int32_t）。
- 变量类型限定符：对于**指针类型**的入参通常需要带 \_\_gm__ 这样的限定符，表明该指针变量指向Global Memory上某处内存地址。为了方便统一，指针入参变量常被定义为 \_\_gm__ uint8_t* 类型，使用时再转为实际类型；也可以直接传入实际指针类型。
- 宏封装：框架提供了 GM_ADDR 宏，用来简化指针入参的书写，避免函数参数列表过长。

    ```
    #define GM_ADDR __gm__ uint8_t* __restrict__
    ```

### 2.3 语法形式
- kernel_name 是自定义的核函数名称。
- (argument list) 是函数的参数列表

---
## 3. Ascend C异构并行编程模型

### 3.1 Host-Device异构协同机制
Ascend C异构计算架构分为Host侧和Device侧（Device侧对应昇腾AI处理器），两者协同完成计算任务。Host侧主要负责运行时管理，包括存储管理、设备管理以及Stream管理等，确保任务的高效调度与资源的合理分配；Device侧，会执行开发者基于Ascend C语法编写的Kernel核函数，主要完成批量数据的矩阵运算、向量运算等计算密集型的任务，用于计算加速。

如下图所示，当一个Kernel下发到AI Core（昇腾AI处理器的计算核心）上执行时，运行时管理模块根据开发者设置的核数和任务类型启动对应的Task，该Task从Host加载到Device的Stream运行队列，调度单元会把就绪的Task分配到空闲AI Core上执行。这里将需要处理的数据拆分并同时在多个计算核心上运行（也就是下文介绍的SPMD并行计算），可以获取更高的性能。


<img src="./images/异构并行编程.png" alt="异构并行编程"  width="700px" >

### 3.2 SPMD并行计算

Ascend C算子编程是SPMD（Single-Program Multiple-Data）编程，通俗来讲就是“一份代码，多处执行，处理的数据不同”。SPMD是一种常用的并行计算的方法，是提高计算速度的有效手段。

具体到Ascend C编程模型中的应用，是将需要处理的数据拆分并同时在多个计算核心上运行，从而获取更高的性能。多个AI Core共享相同的指令代码，每个核上的运行实例唯一的区别是block_idx不同，每个核通过不同的block_idx来识别自己的身份。block的概念类似于进程的概念，block_idx就是标识进程唯一性的进程ID。并行计算过程如下图所示。

<img src="./images/SPMD并行计算示意图.png" alt="SPMD并行计算示意图"  width="700px" >

---
## 4. 如何调用核函数

核函数的调用语句是C/C++函数调用语句的一种扩展，常见的C/C++函数调用方式是如下的形式：

function_name(argument list);

核函数使用内核调用符<<<...>>>这种语法形式，来规定核函数的执行配置：

kernel_name<<<blockDim, l2ctrl, stream>>>(argument list);

参数说明：

- blockDim，规定了核函数将会在几个核上执行，每个执行该核函数的核会被分配一个逻辑ID，表现为内置变量block_idx，编号从0开始，可为不同的逻辑核定义不同的行为，可以在算子实现中使用GetBlockIdx()函数来获得。
- l2ctrl，保留参数，暂时设置为固定值nullptr。
- stream，类型为aclrtStream，stream是一个任务队列，应用程序通过stream来管理任务的并行。

---
## 5. 跑通第一个Ascend C程序————打印Hello World

### 5.1 编写"Hello World"程序

使用Ascend C编写一个简单的"Hello World"程序，在代码文件hello_world.asc中包括核函数实现和主函数实现。

- 核函数实现：该核函数的核心逻辑是输出"Hello World!!!"字符串。
- 主函数实现：在主函数中，进行初始化环境、申请资源、通过<<<>>>调用核函数以及释放资源等操作。

#### 5.1.1 包含头文件及引入命名空间

代码文件hello_world.asc的开头需要引入依赖头文件，包括Host侧应用程序需要包含的头文件和Kernel侧核函数需要包含的头文件。

In [ ]:
%%writefile Sources/hello_world.asc

// Host侧应用程序需要包含的头文件
#include "acl/acl.h"
// Kernel侧需要包含的头文件
#include "kernel_operator.h"

#### 5.1.2 实现核函数

自定义核函数hello_world，函数体内调用Ascend C接口：
- AscendC::printf：提供设备侧调试场景下的格式化输出功能。
- AscendC::GetBlockIdx：获取当前AICORE的逻辑ID
- AscendC::GetBlockNum：获取当前任务配置的AICORE执行核数

In [ ]:
%%writefile -a Sources/hello_world.asc

__global__ __aicore__ void hello_world()
{
 
    AscendC::printf("[Block (%lu/%lu)]: Hello World!!!\n", AscendC::GetBlockIdx(), AscendC::GetBlockNum());
}

#### 5.1.3 调用核函数

完成Kernel侧核函数开发后，即可编写Host侧的核函数调用程序。在一个完整的Ascend C核函数调用过程中，需要执行以下步骤：
- 初始化AscendCL。
- 申请运行管理资源：device、context、stream。
- 申请Host和Device侧的内存，在Host侧读入数据后，将数据拷贝到Device侧。
- 调用<<<..>>>在设定的AICORE上执行核函数。
- 将数据拷贝回Host侧内存。
- 释放运行管理资源：device、context、stream。
- 去初始化AscendCL。

其逻辑流程图如下所示：

<img src="./images/调用流程.png" alt="调用流程"  width="200px" >

这里实现从Host侧的APP程序调用核函数，在8个AICORE上执行打印操作。没有数据在host侧和device侧的传输过程。具体代码如下：

In [ ]:
%%writefile -a Sources/hello_world.asc

int32_t main(int argc, char const *argv[])
{
    // 初始化AscendCL
    aclInit(nullptr);
    // 申请运行管理资源
    int32_t deviceId = 0;
    aclrtSetDevice(deviceId);
    aclrtStream stream = nullptr;
    aclrtCreateStream(&stream);
    // 设置参与计算的AICORE核数为8（核数可根据实际需求设置）
    constexpr uint32_t blockDim = 8;
    // 用内核调用符<<<>>>调用核函数
    hello_world<<<blockDim, nullptr, stream>>>();
    // 调用aclrtSynchronizeStream函数来强制主机端程序等待所有AICORE上的核函数执行完毕。
    aclrtSynchronizeStream(stream);
    // 释放运行管理资源
    aclrtDestroyStream(stream);
    aclrtResetDevice(deviceId);
    // 去初始化AscendCL
    aclFinalize();
    return 0;
}

#### 5.1.4 编译运行

代码开发完成后，通过bisheng命令行将源文件hello_world.asc编译为当前平台的可执行文件：




In [ ]:
# bisheng [算子源文件] --npu-arch=[NPU架构版本号] -o [输出产物名称] 
!bisheng Sources/hello_world.asc --npu-arch=dav-2201 -o demo

再执行以下代码，执行HelloWorld程序：

In [ ]:
!./demo

---
## 课后实践

请根据本节教程内容，在下面的代码框中补充实现 hi_ascend.asc 核函数，使其能够在 16 个 AICore 上并行打印“Hi Ascend”。









In [ ]:
%%writefile Sources/hi_ascend.asc

// Host侧应用程序需要包含的头文件
#include "acl/acl.h"
// Kernel侧需要包含的头文件
#include "kernel_operator.h"

__global__ __aicore__ void hi_ascend()
{
    // 待补充……
}

int32_t main(int argc, char const *argv[])
{
    aclInit(nullptr);
    int32_t deviceId = 0;
    aclrtSetDevice(deviceId);
    aclrtStream stream = nullptr;
    aclrtCreateStream(&stream);
    // 待补充……
    aclrtSynchronizeStream(stream);
    aclrtDestroyStream(stream);
    aclrtResetDevice(deviceId);
    aclFinalize();
    return 0;
}


完成 hi_ascend.asc 核函数的开发后，执行以下命令进行编译并验证结果：

In [ ]:
!bisheng Sources/hi_ascend.asc --npu-arch=dav-2201 -o demo
!./demo

预期执行效果如下：

<img src="./images/执行结果02.png" alt="执行结果02"  width="800px" >

执行以下代码获取答案。

In [ ]:
!cat ./answer/02.02_answer.txt